# 📦 01 - Pick-Place Expert Dataset Generation

This notebook generates expert demonstrations for MetaWorld `pick-place-v3` using the proper **LeRobot dataset format**.

**Key Steps:**
1. Install dependencies
2. 🔍 DEBUG: Test scripted expert
3. Collect successful demonstrations using `LeRobotDataset.create()`
4. Upload to HuggingFace with `dataset.push_to_hub()`

In [ ]:
# ==========================================
# CELL 1: SYSTEM DEPENDENCIES
# ==========================================

!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
                         libosmesa6-dev software-properties-common patchelf

print("✅ System dependencies installed")

In [ ]:
# ==========================================
# CELL 2: ENVIRONMENT SETUP
# ==========================================

import os
import sys

os.environ['MUJOCO_GL'] = 'egl'
os.environ['LEROBOT_VIDEO_BACKEND'] = 'pyav'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    print("✅ HF_TOKEN loaded")
except:
    print("⚠️ Set HF_TOKEN manually: os.environ['HF_TOKEN'] = 'your_token'")

print("✅ Environment configured")

In [ ]:
# ==========================================
# CELL 3: INSTALL PACKAGES
# ==========================================

!git clone https://github.com/huggingface/lerobot.git /kaggle/working/lerobot 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/lerobot
!pip install -e . -q
!pip install metaworld imageio imageio-ffmpeg av -q

print("\n✅ All packages installed")

In [ ]:
# ==========================================
# CELL 4: CONFIGURATION
# ==========================================

import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/lerobot/src")

# Task configuration
TASK_NAME = "pick-place-v3"
NUM_EPISODES = 50
MAX_EPISODE_STEPS = 500
OBSERVATION_WIDTH = 480
OBSERVATION_HEIGHT = 480
FPS = 20

# Paths
ROOT_DIR = Path("/kaggle/working/data")
ROOT_DIR.mkdir(parents=True, exist_ok=True)

# HuggingFace - CHANGE THIS!
HF_USERNAME = "aryannzzz"  # <-- Your HuggingFace username
DATASET_REPO_ID = f"{HF_USERNAME}/metaworld-{TASK_NAME}-expert-v2"

print("📋 Configuration:")
print(f"   Task: {TASK_NAME}")
print(f"   Episodes: {NUM_EPISODES}")
print(f"   Resolution: {OBSERVATION_WIDTH}x{OBSERVATION_HEIGHT}")
print(f"   HF Repo: {DATASET_REPO_ID}")

---
## 🔍 DEBUG: Test Scripted Expert

In [ ]:
# ==========================================
# CELL 5: 🔍 DEBUG - Test Scripted Expert
# ==========================================

from lerobot.envs.metaworld import MetaworldEnv
import numpy as np

print("🔍 Testing scripted expert policy...")

# Create environment
env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=OBSERVATION_WIDTH,
    observation_height=OBSERVATION_HEIGHT,
)

# Get scripted policy
from metaworld.policies import SawyerPickPlaceV2Policy
expert_policy = SawyerPickPlaceV2Policy()

# Test one episode
obs, info = env.reset(seed=42)
success = False
total_reward = 0

for step in range(200):
    # Get raw observation for expert policy
    raw_obs = env._env._get_obs()
    action = expert_policy.get_action(raw_obs)
    action = np.clip(action, -1.0, 1.0).astype(np.float32)
    
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    if info.get('success', False) or info.get('is_success', False):
        success = True
        print(f"   ✅ SUCCESS at step {step+1}!")
        break

env.close()

print(f"\n�� Test Result:")
print(f"   Reward: {total_reward:.2f}")
print(f"   Success: {success}")

if success:
    print("\n✅ Expert works! Ready to collect dataset.")
else:
    print("\n❌ Expert failed! Check MetaWorld installation.")

---
## �� Generate Dataset with LeRobot API

In [ ]:
# ==========================================
# CELL 6: CREATE LEROBOT DATASET
# ==========================================

import shutil
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.envs.metaworld import MetaworldEnv, TASK_DESCRIPTIONS

# Local repo ID for creation
LOCAL_REPO_ID = f"lerobot/{TASK_NAME}"
dataset_path = ROOT_DIR / LOCAL_REPO_ID

# Clean up existing dataset
if dataset_path.exists():
    print(f"⚠️ Removing existing dataset at {dataset_path}")
    shutil.rmtree(dataset_path)

# Define features for LeRobot dataset
features = {
    "observation.images.image": {
        "dtype": "video",
        "shape": (OBSERVATION_HEIGHT, OBSERVATION_WIDTH, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (4,),
        "names": ["x", "y", "z", "gripper"],
    },
    "action": {
        "dtype": "float32",
        "shape": (4,),
        "names": ["dx", "dy", "dz", "gripper"],
    },
    "next.reward": {
        "dtype": "float32",
        "shape": (1,),
        "names": ["reward"],
    },
    "next.success": {
        "dtype": "bool",
        "shape": (1,),
        "names": ["success"],
    },
}

# Create dataset with proper LeRobot format
print(f"📦 Creating LeRobot dataset...")
dataset = LeRobotDataset.create(
    repo_id=LOCAL_REPO_ID,
    fps=FPS,
    root=dataset_path,
    features=features,
    robot_type="sawyer",
    use_videos=True,
)

print(f"✅ Dataset created at {dataset_path}")

In [ ]:
# ==========================================
# CELL 7: RECORD EXPERT DEMONSTRATIONS
# ==========================================

from metaworld.policies import SawyerPickPlaceV2Policy
import numpy as np

print(f"\n{'='*70}")
print(f"🎬 Recording Expert Demonstrations")
print(f"{'='*70}")
print(f"   Task: {TASK_NAME}")
print(f"   Target episodes: {NUM_EPISODES}")
print(f"{'='*70}\n")

# Create environment
env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=OBSERVATION_WIDTH,
    observation_height=OBSERVATION_HEIGHT,
)

expert_policy = SawyerPickPlaceV2Policy()
task_description = TASK_DESCRIPTIONS.get(TASK_NAME, f"perform {TASK_NAME}")

# Recording stats
success_count = 0
total_attempts = 0
total_frames = 0

print("🚀 Starting recording...\n")

while success_count < NUM_EPISODES:
    total_attempts += 1
    seed = np.random.randint(0, 1_000_000)
    
    print(f"🎬 Episode attempt {total_attempts} (seed={seed})...", end=" ")
    
    obs, info = env.reset(seed=seed)
    dataset.episode_buffer = dataset.create_episode_buffer()
    
    done = False
    step_count = 0
    episode_success = False
    
    while not done and step_count < MAX_EPISODE_STEPS:
        image = obs["pixels"]
        agent_pos = obs["agent_pos"]
        
        # Get expert action using raw internal state
        raw_obs = env._env._get_obs()
        action = expert_policy.get_action(raw_obs)
        action = np.clip(action, -1.0, 1.0).astype(np.float32)
        
        next_obs, reward, terminated, truncated, step_info = env.step(action)
        done = terminated or truncated
        
        if step_info.get("is_success", False) or step_info.get("success", 0) > 0.5:
            episode_success = True
        
        # Add frame to dataset
        frame = {
            "observation.images.image": image,
            "observation.state": agent_pos.astype(np.float32),
            "action": action,
            "next.reward": np.array([reward], dtype=np.float32),
            "next.success": np.array([episode_success]),
            "task": task_description,
        }
        dataset.add_frame(frame)
        
        obs = next_obs
        step_count += 1
    
    # Only save SUCCESSFUL episodes
    if episode_success:
        success_count += 1
        total_frames += step_count
        dataset.save_episode()  # CRITICAL: This saves metadata!
        print(f"✅ SUCCESS ({success_count}/{NUM_EPISODES}) - {step_count} frames")
    else:
        print(f"❌ FAILED")
        dataset.episode_buffer = dataset.create_episode_buffer()
    
    # Safety limit
    if total_attempts > NUM_EPISODES * 5:
        print(f"\n⚠️ Too many failed attempts. Stopping.")
        break

env.close()

print(f"\n{'='*70}")
print(f"📊 Recording Summary")
print(f"{'='*70}")
print(f"   Successful episodes: {success_count}")
print(f"   Total attempts: {total_attempts}")
print(f"   Success rate: {100 * success_count / total_attempts:.1f}%")
print(f"   Total frames: {total_frames:,}")
print(f"   Avg frames/episode: {total_frames / max(success_count, 1):.1f}")
print(f"{'='*70}")

In [ ]:
# ==========================================
# CELL 8: 🔍 DEBUG - Verify Local Dataset
# ==========================================

print("🔍 Verifying local dataset...\n")

# Check files exist
meta_dir = dataset_path / "meta"
required_files = ["info.json", "stats.json", "episodes.jsonl"]

print("📁 Checking metadata files:")
all_present = True
for f in required_files:
    path = meta_dir / f
    if path.exists():
        print(f"   ✅ {f}")
    else:
        print(f"   ❌ {f} MISSING!")
        all_present = False

if all_present:
    print("\n✅ All metadata files present!")
else:
    print("\n❌ Some metadata files missing. Dataset may not load correctly.")

# Load and verify
print(f"\n📊 Dataset Statistics:")
print(f"   Episodes: {dataset.num_episodes}")
print(f"   Total frames: {dataset.num_frames:,}")
print(f"   FPS: {dataset.fps}")

# Check a sample
if dataset.num_frames > 0:
    sample = dataset[0]
    print(f"\n📐 Sample frame:")
    for key, val in sample.items():
        if hasattr(val, 'shape'):
            print(f"   {key}: {val.shape}")

In [ ]:
# ==========================================
# CELL 9: UPLOAD TO HUGGINGFACE
# ==========================================

print(f"\n{'='*70}")
print(f"📤 Uploading Dataset to HuggingFace")
print(f"{'='*70}")
print(f"   Repository: {DATASET_REPO_ID}")
print(f"{'='*70}\n")

# Reload dataset with target repo_id
dataset_for_upload = LeRobotDataset(
    repo_id=DATASET_REPO_ID,
    root=dataset_path,
    video_backend="pyav"
)

# Push to hub - this uploads with all metadata!
dataset_for_upload.push_to_hub(
    tags=["metaworld", "pick-place", "expert", "lerobot"],
    license="apache-2.0",
    push_videos=True,
)

print(f"\n{'='*70}")
print(f"✅ Dataset uploaded successfully!")
print(f"{'='*70}")
print(f"\n🔗 View at: https://huggingface.co/datasets/{DATASET_REPO_ID}")

In [ ]:
# ==========================================
# CELL 10: 🔍 DEBUG - Verify Upload
# ==========================================

from huggingface_hub import list_repo_files

print(f"🔍 Verifying uploaded dataset: {DATASET_REPO_ID}\n")

try:
    files = list_repo_files(DATASET_REPO_ID, repo_type="dataset")
    
    print("📁 Files on HuggingFace:")
    for f in sorted(files)[:15]:
        print(f"   {f}")
    if len(files) > 15:
        print(f"   ... and {len(files) - 15} more files")
    
    # Check for required metadata
    required = ['meta/info.json', 'meta/stats.json', 'meta/episodes.jsonl']
    missing = [f for f in required if f not in files]
    
    if not missing:
        print("\n✅ All LeRobot metadata files present!")
        print("✅ Dataset ready for training with 02_pickplace_train.ipynb")
    else:
        print(f"\n❌ Missing files: {missing}")
        
except Exception as e:
    print(f"❌ Error: {e}")

---
## ✅ Dataset Generation Complete!

**Next Steps:**
1. Run **02_pickplace_train.ipynb** to train the ACT policy

**Expected Output:**
- 50 successful episodes
- ~2,500-3,000 frames total
- All metadata files present on HuggingFace